## DuckDB Cloud Storage Sample

This sample shows how to access a cloud storage from DuckDB.

It uses AWS S3 (MinIO), but you can access Cloudflare R2 and GCS in the same way.

### Setting up

In [1]:
import os
from pathlib import Path

import duckdb

### Connecting to DuckDB database

In [ ]:
db = duckdb.connect(
    database=Path(os.environ["SD_DATA_DIR"]) / "sample.db",
    read_only=False,
)

### Loading DuckDB extensions

In [3]:
db.install_extension("httpfs")

db.load_extension("httpfs")

### Creating DuckDB secret

In this sample, it creates a secret to access AWS S3 (MinIO) with an access key pair.

If you are using other cloud storages, refer to the following sample and [documentation](https://duckdb.org/docs/extensions/httpfs/s3api.html).


#### Samples


##### AWS S3 with Access Key Pair

```sql
CREATE OR REPLACE SECRET sample (
    TYPE S3,
    REGION '<Your Region>',
    KEY_ID '<Your Access Key>',
    SECRET  '<Your Secret Access Key>'
);
```


##### AWS S3 with Credential Provider Chain

````sql
CREATE OR REPLACE SECRET sample (
    TYPE S3,
    REGION 'us-east-1',
    PROVIDER CREDENTIAL_CHAIN,
    CHAIN 'config',
    PROFILE '<Your AWS Profile>'
);
````


##### GCS

```sql
CREATE OR REPLACE SECRET sample (
    TYPE GCS,
    KEY_ID '<Your Access Key>',
    SECRET  '<Your Secret>'
);
```

In [4]:
s = """
    CREATE OR REPLACE SECRET sample (
        TYPE S3,
        REGION 'us-east-1',
        KEY_ID '%(s3_access_key_id)s',
        SECRET  '%(s3_secret_access_key)s',
        ENDPOINT 'localhost:%(s3_port)s',
        URL_STYLE 'path',
        USE_SSL FALSE
    );
""" % {
    "s3_access_key_id": os.environ["MINIO_ROOT_USER"],
    "s3_secret_access_key": os.environ["MINIO_ROOT_PASSWORD"],
    "s3_port": os.environ["MINIO_PORT"],
}
_ = db.execute(s)

### Reading data from cloud storage

In [ ]:
db.sql("FROM 's3://default/samples/first.jsonl'")

### Creating table from files on AWS S3

In [6]:
db.sql("CREATE OR REPLACE TABLE tmp_duckdb_with_aws_s3 AS FROM 's3://default/samples/first.jsonl'")

### Cleaning up

In [8]:
db.close()